<a href="https://colab.research.google.com/github/memin01/film-sitesi/blob/main/plaka_tespiti.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Plaka Bölgesi Tespiti: Kenar ve Morfoloji Tabanlı Yaklaşım

Bu çalışma, **klasik görüntü işleme** teknikleriyle (derin öğrenme **kullanmadan**) bir araç resminden Türk plakasını tespit eden bir pipeline kurar. Yaklaşımın ana adımları:

| Adım | Teknik | Amaç |
|---|---|---|
| 1 | Gri tonlama + Bilateral filtre + CLAHE | Gürültü azaltma, kontrast iyileştirme |
| 2 | Sobel-X (+ BlackHat) | Plaka karakterlerinin oluşturduğu dikey kenarları öne çıkarmak |
| 3 | Otsu eşikleme | İkili görüntüye çevirme |
| 4 | Morfolojik kapama | Bitişik karakterleri tek bir "blok" haline getirmek |
| 5 | Kontur + geometrik filtre | Plaka oranına (en/boy ≈ 4.5) uyan dikdörtgenleri seçmek |
| 6 | Tesseract OCR + regex | Plakayı okuyup Türk plaka formatına uydurmak |

Türk plakası fiziksel özellikleri:
- Boyut: **520 mm × 110 mm** → en/boy oranı ≈ **4.7**
- Beyaz arka plan, koyu karakterler
- Format: `İL[2 rakam] [1-3 harf] [2-4 rakam]` (toplam 7 ya da 8 karakter)


## 1. Kurulum

Gereken paketler: `opencv-python`, `numpy`, `matplotlib`, `pytesseract`.  
Ayrıca sistemde **Tesseract OCR** binary'si kurulu olmalı.

Aşağıdaki hücre **Colab veya temiz bir ortamda** eksik olanları otomatik yükler.

- **Colab / Linux**: `apt-get install tesseract-ocr` + `pip install pytesseract`
- **macOS**: `brew install tesseract` + `pip install pytesseract`
- **Windows**: Tesseract binary'sini [UB Mannheim build](https://github.com/UB-Mannheim/tesseract/wiki)'inden indir, sonra `pytesseract.pytesseract.tesseract_cmd` değişkenini exe yoluna ayarla


In [ ]:
# ---- Kurulum (Colab veya temiz ortam için) ----
# Tesseract OCR binary'si + pytesseract Python wrapper'ı
import sys, subprocess, shutil

# 1) Sistem paketi: tesseract-ocr (Linux / Colab)
if shutil.which("tesseract") is None:
    print("Tesseract binary kuruluyor...")
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(["apt-get", "-qq", "install", "-y", "tesseract-ocr"], check=False)
    # macOS:  brew install tesseract
    # Windows: https://github.com/UB-Mannheim/tesseract/wiki adresinden indir

# 2) Python paketleri
for paket in ["opencv-python", "pytesseract"]:
    try:
        __import__(paket.replace("-python", "").replace("-", "_"))
    except ImportError:
        print(f"{paket} kuruluyor...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paket], check=False)

# ---- Importlar ----
import os
import re
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pytesseract

# Windows kullanıcısıysan tesseract.exe yolunu burada belirt:
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

print(f"OpenCV   : {cv2.__version__}")
print(f"NumPy    : {np.__version__}")
print(f"Tesseract: {pytesseract.get_tesseract_version()}")

opencv-python kuruluyor...
OpenCV   : 4.13.0
NumPy    : 2.0.2
Tesseract: 4.1.1


## 2. Yardımcı: Görselleştirme

Ara adımları yan yana göstermek için küçük bir yardımcı fonksiyon.


In [ ]:
def goster(*resimler_ve_basliklar, figsize=(15, 5)):
    n = len(resimler_ve_basliklar)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for ax, (img, baslik) in zip(axes, resimler_ve_basliklar):
        if img is None:
            ax.axis("off"); continue
        if len(img.shape) == 2:
            ax.imshow(img, cmap="gray")
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(baslik); ax.axis("off")
    plt.tight_layout(); plt.show()

## 3. Ön İşleme

Plaka bölgesindeki kenarları tespit edebilmek için resmi:
1. **Gri tonlamaya** çeviririz (renk bilgisi gerekli değil)
2. **Bilateral filtre** uygularız: kenarları korurken pürüzleri yumuşatır (gauss filtresinin aksine plaka sınırı bulanmaz)
3. **CLAHE** (Contrast Limited Adaptive Histogram Equalization) ile lokal kontrastı iyileştiririz — özellikle düşük ışıkta plakanın belirginleşmesini sağlar


In [ ]:
def on_isleme(bgr):
    # BGR resmi gri tonlama + gürültü azaltma + CLAHE'ye sokar.
    gri = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    # Bilateral: d=komşuluk yarıçapı, sigmaColor/sigmaSpace: renk ve uzay benzerliği
    blur = cv2.bilateralFilter(gri, d=11, sigmaColor=17, sigmaSpace=17)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return gri, clahe.apply(blur)

## 4. Kenar Tespiti

Plaka karakterleri **yoğun dikey kenarlar** üretir (rakamların kenarları çoğunlukla dik). Bu yüzden Sobel operatörünün **X türevi** plakanın doku imzasını çıkarmak için idealdir.

İki yardımcı kenar haritası hesaplıyoruz:
- **Sobel-X**: dikey kenarları vurgular (asıl ipucu)
- **BlackHat**: koyu zemin üzerinde küçük parlak yapıları öne çıkarır. *Ama biz beyaz plakada koyu karakter arıyoruz; o yüzden burada BlackHat'ı gri ton üzerinde değil, ters versiyonu (TopHat'a benzer) için kullanırız — opsiyonel bir destekçi maske.*


In [ ]:
def kenar_haritasi(gri):
    # Sobel-X + Otsu ikili maske.
    sobelx = cv2.Sobel(gri, cv2.CV_8U, dx=1, dy=0, ksize=3)
    _, esik = cv2.threshold(sobelx, 0, 255,
                            cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return sobelx, esik

def blackhat_haritasi(gri, kernel_w=13, kernel_h=5):
        # BlackHat: kapalı görüntü - orijinal -> koyu yapıların öne çıkması.
    # Türk plakası beyaz zeminde koyu karakterlere sahip; karakter blokları
    # burada zayıf da olsa belirgin olur.
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_w, kernel_h))
    bh = cv2.morphologyEx(gri, cv2.MORPH_BLACKHAT, k)
    _, esik = cv2.threshold(bh, 0, 255,
                            cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return bh, esik

## 5. Morfolojik Kapama (Closing)

Sobel sonrası elimizde **dağınık dikey çizgiler** var. Aynı plakanın karakterleri yan yana olduğu için bu çizgileri **yatay yönde birleştirip** tek bir dikdörtgen bloğa dönüştürmek istiyoruz.

Bunun için:
- **Yatay yapısal element** (örn. 22×5): yan yana ama bağlantısız yapıları köprülemek için
- **MORPH_CLOSE**: önce dilation, sonra erosion (gap kapatır)
- Küçük bir **opening** ile noise temizliği

**Neden tek bir kernel boyutu yetmez?**  
Kameraya yakın bir plaka 200 px geniş olabilirken uzaktaki bir plaka 80 px olabilir. Tek bir kapama kerneliyle her ikisini de doğru birleştiremeyiz. Bu yüzden **birden fazla kernel boyutu** deneyip sonuçları birleştiririz.


In [ ]:
def morfolojik_kapama(esik_maske, kernel_boyutlari=((13, 5), (22, 5), (30, 7), (45, 9))):
    # Birden fazla yatay kernel ile kapama; tüm maskeleri OR ile birleştirir.
    maskeler = []
    kucuk_k = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    for kw, kh in kernel_boyutlari:
        k = cv2.getStructuringElement(cv2.MORPH_RECT, (kw, kh))
        # Closing: dikey karakterleri yatayda birleştirir
        m = cv2.morphologyEx(esik_maske, cv2.MORPH_CLOSE, k)
        # Opening: küçük gürültüleri at
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN, kucuk_k)
        # Hafif dilation: kontur kenarlarını koru
        m = cv2.dilate(m, kucuk_k, iterations=1)
        maskeler.append(m)
    return maskeler

## 6. Plaka Adaylarını Bulma

Maskelerden konturları çıkardıktan sonra **plakaya benzemeyenleri elemek** için geometrik kurallar uygularız:

| Kural | Değer | Neden? |
|---|---|---|
| En/boy oranı | 2.0 – 7.5 | Türk plakası ideal ~4.5; eğik açıdan biraz dar görünebilir |
| Minimum boyut | w ≥ 60, h ≥ 15 | Çok küçük bloklar OCR'a yetmez |
| Resme alan oranı | %0.1 – %25 | Çok küçükse gürültü, çok büyükse arka plan |
| Ortalama parlaklık | ≥ 70 | Plaka beyaz zemine sahip; koyu bloklar (BMW grilliği vb.) elenir |
| Parlaklık std. sapması | ≥ 25 | Düz (yansıma vb.) bölgeleri ele; metin içeren bölgeler yüksek std verir |

Her aday **bir skor** alır (oran-iyiliği × doluluk × parlaklık) ve azalan sırayla sıralanır.


In [ ]:
def plaka_adaylari(maskeler, gri):
    H, W = gri.shape
    img_alan = H * W
    adaylar = []
    gorulmus = set()  # dedup

    for mask in maskeler:
        konturlar, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)
        for c in konturlar:
            x, y, w, h = cv2.boundingRect(c)
            if h == 0 or w < 60 or h < 15:
                continue
            oran = w / h
            alan_or = (w * h) / img_alan
            if not (2.0 <= oran <= 7.5):           continue
            if not (0.001 <= alan_or <= 0.25):     continue

            # Parlaklık filtresi (beyaz plaka heuristic)
            bolge = gri[y:y+h, x:x+w]
            ort = float(np.mean(bolge))
            std = float(np.std(bolge))
            if ort < 70 or std < 25:
                continue

            doluluk = cv2.contourArea(c) / (w * h)
            # Dedup: yakın bbox'lar zaten görüldüyse atla
            anahtar = (x // 15, y // 15, w // 15, h // 15)
            if anahtar in gorulmus:
                continue
            gorulmus.add(anahtar)

            # Bileşik skor
            oran_skor = 1.0 - abs(oran - 4.5) / 4.5
            parlak_bonus = min(1.0, ort / 200.0)
            skor = oran_skor * doluluk * (0.5 + 0.5 * parlak_bonus)

            adaylar.append({
                "bbox": (x, y, w, h),
                "oran": oran, "alan": w * h,
                "doluluk": doluluk, "parlaklik": ort,
                "skor": skor,
            })
    adaylar.sort(key=lambda d: d["skor"], reverse=True)
    return adaylar

## 7. OCR ile Karakter Okuma

Aday bulunduktan sonra Tesseract OCR'a göndermeden önce plakayı temizleriz:
- **Büyüt**: Tesseract en az ~50 px yükseklikte karakter ister
- **Otsu eşikleme** ile temiz ikili görüntü
- Eğer karakterler beyazsa **invert** et (Tesseract koyu metin bekler)

OCR'da:
- `--psm 7`: "tek satır metin" varsayımı (plaka tek satır)
- `--psm 8`: "tek kelime" — alternatif
- `tessedit_char_whitelist`: yalnızca büyük harf ve rakam


In [ ]:
def ocr_icin_hazirla(plaka_bgr, mod="otsu"):
    if plaka_bgr is None or plaka_bgr.size == 0:
        return None
    h, w = plaka_bgr.shape[:2]
    # Tesseract için minimum yükseklik
    if h < 60:
        scale = 100.0 / h
        plaka_bgr = cv2.resize(plaka_bgr, (int(w * scale), 100),
                               interpolation=cv2.INTER_CUBIC)
    gri = (cv2.cvtColor(plaka_bgr, cv2.COLOR_BGR2GRAY)
           if len(plaka_bgr.shape) == 3 else plaka_bgr)
    gri = cv2.bilateralFilter(gri, 11, 17, 17)

    if mod == "otsu":
        _, ikili = cv2.threshold(gri, 0, 255,
                                 cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    elif mod == "adaptive":
        ikili = cv2.adaptiveThreshold(gri, 255,
                                      cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                      cv2.THRESH_BINARY, 31, 10)
    else:
        ikili = gri

    # Tesseract koyu metin bekler. Eğer ortalama < 127 ise zaten siyah zemin
    # üzerinde beyaz metin var demektir; ters çevir.
    if np.mean(ikili) < 127:
        ikili = cv2.bitwise_not(ikili)
    return ikili

def ocr_oku(ikili, psm=7):
    if ikili is None:
        return ""
    cfg = (f"--psm {psm} --oem 3 "
           f"-c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789")
    try:
        return pytesseract.image_to_string(ikili, config=cfg, lang="eng").strip()
    except Exception:
        return ''


## 8. OCR Çıktısını Türk Plaka Formatına Uydurma

Tesseract bize kirli bir string verir: `"B34LE89681"` gibi. Bunu Türk plaka formatına oturtmalıyız.

**Türk plakası kuralları:**
- İl kodu: 01-81 arası iki haneli sayı
- Harf bloğu: 1-3 büyük harf
- Sıra numarası: 2-4 rakam
- Toplam: 7 ya da 8 karakter

**Çoğunlukla OCR şu hataları yapar:**
| Yanlış okunan | Doğrusu | Neden |
|---|---|---|
| `B` | `8` (rakam bölgesinde) | Benzer şekil |
| `O` | `0` | Benzer şekil |
| `I` | `1` | Benzer şekil |
| `S` | `5` | Benzer şekil |
| `Z` | `2` | Benzer şekil |

Önce orijinal metni tüm formatlarda dene, eşleşme yoksa **karakter karışıklık varyantlarını** üretip tekrar dene.


In [ ]:
# Türk plakası format kalıpları (uzun -> kısa, 8 karakter öncelikli)
TR_FORMATS = [
    re.compile(r"(0[1-9]|[1-7]\d|8[01])([A-Z]{2})(\d{4})"),   # 8: 34 AB 1234
    re.compile(r"(0[1-9]|[1-7]\d|8[01])([A-Z]{3})(\d{3})"),   # 8: 34 ABC 123
    re.compile(r"(0[1-9]|[1-7]\d|8[01])([A-Z])(\d{5})"),      # 8: 34 A 12345
    re.compile(r"(0[1-9]|[1-7]\d|8[01])([A-Z]{2})(\d{3})"),   # 7: 34 AB 123
    re.compile(r"(0[1-9]|[1-7]\d|8[01])([A-Z]{3})(\d{2})"),   # 7: 34 ABC 12
    re.compile(r"(0[1-9]|[1-7]\d|8[01])([A-Z])(\d{4})"),      # 7: 34 A 1234
]

HARF_BENZERI = {"8": "B", "0": "O", "1": "I", "5": "S", "2": "Z"}
RAKAM_BENZERI = {"B": "8", "O": "0", "I": "1", "S": "5", "Z": "2", "D": "0"}

def varyantlari_uret(metin):
        # Karakter karışıklığı varyantlarını üretir. Harf bölgesinde rakam->harf,
    # rakam bölgesinde harf->rakam dönüşümleri.
    if not metin:
        return []
    varyantlar = [metin]
    n = len(metin)
    # Harf bölgesi (poz. 2-4): rakam -> harf
    for i in range(2, min(5, n)):
        if metin[i] in HARF_BENZERI:
            varyantlar.append(metin[:i] + HARF_BENZERI[metin[i]] + metin[i+1:])
    # Rakam bölgesi (son 4-5): harf -> rakam
    for i in range(max(0, n-5), n):
        if metin[i] in RAKAM_BENZERI:
            varyantlar.append(metin[:i] + RAKAM_BENZERI[metin[i]] + metin[i+1:])
    # Tekilleştir, sırayı koru
    return list(dict.fromkeys(varyantlar))

def plaka_formatla(ham_metin):
    # OCR çıktısından Türk plakası çıkarır, yoksa None döner.
    if not ham_metin:
        return None
    # Yalnızca harf-rakam karakterlere indir
    temiz = re.sub(r"[^A-Z0-9]", "", ham_metin.upper())
    # Plakanın solunda bulunabilecek 'TR' önekini at
    if temiz.startswith("TR"):
        temiz = temiz[2:]
    if not temiz:
        return None

    # Önce orijinali tüm formatlarda dene (uzun format öncelikli)
    for pat in TR_FORMATS:
        m = pat.search(temiz)
        if m:
            return f"{m.group(1)} {m.group(2)} {m.group(3)}"

    # Eşleşme yoksa varyantları dene
    for varyant in varyantlari_uret(temiz):
        if varyant == temiz:
            continue
        for pat in TR_FORMATS:
            m = pat.search(varyant)
            if m:
                return f"{m.group(1)} {m.group(2)} {m.group(3)}"
    return None

## 9. Tüm Adımları Birleştiren Ana Pipeline

`plaka_tespit_et()` tüm adımları sırayla çağırır ve bulduğu plakayı döndürür.

Önemli iki nokta:
1. **Çoklu OCR denemesi**: Her aday için Otsu + Adaptive eşikleme × PSM 7/8/6 kombinasyonları denenir, Türk plaka formatına uyan ilk eşleşmede dur.
2. **Yedek geri dönüş**: Format uyan bulunamazsa, en yüksek skorlu adayın ham OCR çıktısı döner ki kullanıcı durumu anlayabilsin.


In [ ]:
def plaka_tespit_et(img_yolu, gorsel_olarak_goster=False, max_aday=15):
    img = cv2.imread(img_yolu)
    if img is None:
        raise FileNotFoundError(f"Resim bulunamadı: {img_yolu}")

    # Büyük resimleri standartlaştır
    H, W = img.shape[:2]
    if W > 1200:
        oran = 1200.0 / W
        img = cv2.resize(img, (1200, int(H * oran)))

    # 1) Ön işleme
    gri_orj, gri = on_isleme(img)
    # 2) Kenar haritaları
    _, sobel_esik = kenar_haritasi(gri)
    _, blackhat_esik = blackhat_haritasi(gri)
    # 3) Morfolojik kapama (her iki kenar haritası için)
    maskeler = (morfolojik_kapama(sobel_esik)
                + morfolojik_kapama(blackhat_esik))
    # 4) Plaka adayları
    adaylar = plaka_adaylari(maskeler, gri_orj)

    # 5) OCR
    en_iyi_metin = None
    en_iyi_bbox = None
    en_iyi_plaka = None
    yedek_metin = None  # format uymazsa ham çıktı

    for aday in adaylar[:max_aday]:
        x, y, w, h = aday["bbox"]
        plaka_img = img[max(0, y-4):y+h+4, max(0, x-4):x+w+4]

        for mod in ("otsu", "adaptive"):
            ikili = ocr_icin_hazirla(plaka_img, mod)
            for psm in (7, 8, 6):
                ham = ocr_oku(ikili, psm)
                formatli = plaka_formatla(ham)
                if formatli:
                    en_iyi_metin = formatli
                    en_iyi_bbox = aday["bbox"]
                    en_iyi_plaka = plaka_img
                    break
                if not yedek_metin and ham and len(ham) >= 5:
                    yedek_metin = re.sub(r"[^A-Z0-9]", "", ham.upper())
            if en_iyi_metin: break
        if en_iyi_metin: break

    if gorsel_olarak_goster:
        vis = img.copy()
        # En iyi adayı çiz
        if en_iyi_bbox is not None:
            x, y, w, h = en_iyi_bbox
            cv2.rectangle(vis, (x, y), (x+w, y+h), (0, 255, 0), 3)
        # Diğer adaylar
        for a in adaylar[:10]:
            x, y, w, h = a["bbox"]
            if a["bbox"] != en_iyi_bbox:
                cv2.rectangle(vis, (x, y), (x+w, y+h), (0, 165, 255), 1)
        goster((img, "Orijinal"),
               (sobel_esik, "Sobel-X + Otsu"),
               (maskeler[1], "Morfolojik kapama"),
               (vis, f"Tespit: {en_iyi_metin or '(format uymadı)'}"))

    return {
        "plaka": en_iyi_metin,
        "yedek_ocr": yedek_metin,
        "bbox": en_iyi_bbox,
        "plaka_img": en_iyi_plaka,
        "aday_sayisi": len(adaylar),
    }

## 10. Test

Pipeline'ı çeşitli resimler üzerinde deneyelim. Aşağıda kendi dosyalarını test etmek için `TEST_RESIMLERI` listesine yol ekleyebilirsin.


In [ ]:
TEST_RESIMLERI = [
    "/mnt/user-data/uploads/a06131661f854ba9ae233e62969588d9.jpg",
    "/mnt/user-data/uploads/plaka-tanima-sistemi-3.png",
    "/mnt/user-data/uploads/plaka-tanima-sistemleri-2.png",
    "/mnt/user-data/uploads/WhatsApp_Image_2026-05-12_at_00_10_53.jpeg",
    "/mnt/user-data/uploads/plaka-tanıma-sistemleri.jpg",
]

for yol in TEST_RESIMLERI:
    if not os.path.exists(yol):
        print(f"[!] Bulunamadı: {yol}");  continue
    s = plaka_tespit_et(yol, gorsel_olarak_goster=True)
    if s["plaka"]:
        print(f"  ➜ Tespit edilen plaka: {s['plaka']}")
    else:
        print(f"  ➜ Plaka tespit edilemedi (yedek OCR: {s['yedek_ocr']})")

[!] Bulunamadı: /mnt/user-data/uploads/a06131661f854ba9ae233e62969588d9.jpg
[!] Bulunamadı: /mnt/user-data/uploads/plaka-tanima-sistemi-3.png
[!] Bulunamadı: /mnt/user-data/uploads/plaka-tanima-sistemleri-2.png
[!] Bulunamadı: /mnt/user-data/uploads/WhatsApp_Image_2026-05-12_at_00_10_53.jpeg
[!] Bulunamadı: /mnt/user-data/uploads/plaka-tanıma-sistemleri.jpg


## 11. Tek Bir Resim Üzerinde Hızlı Test

Aşağıdaki hücrede sadece bir dosya yolu vererek hızlıca test edebilirsin. Tespit edilen plaka **ekrana yazdırılır.**


In [ ]:
RESIM_YOLU = "/content/1.jpg"

sonuc = plaka_tespit_et(RESIM_YOLU, gorsel_olarak_goster=True)

print("=" * 50)
if sonuc["plaka"]:
    print(f"TESPİT EDİLEN PLAKA: {sonuc['plaka']}")
else:
    print(f"Plaka tespit edilemedi.")
    if sonuc["yedek_ocr"]:
        print(f"OCR'ın okuduğu en yakın metin: {sonuc['yedek_ocr']}")
print("=" * 50)

FileNotFoundError: Resim bulunamadı: /content/1.jpg

## 12. Yöntemin Sınırları

Kenar + morfoloji tabanlı yaklaşım **basit, hızlı ve eğitim amaçlı çok değerlidir**, ama bazı kısıtları vardır:

- **Yansıma, filigran ve grilliği güçlü olan araçlar**: Beyaz plakayı domine eden kenarlar yanıltabilir
- **Eğik açıdan çekim**: Plaka oranı bozulur, geometrik filtre tutmayabilir → perspektif düzeltme eklenebilir
- **Çok uzaktan / gece çekimleri**: Düşük çözünürlükte OCR zayıflar
- **OCR karakter karışıklıkları**: `8` ↔ `B`, `0` ↔ `O` gibi. Regex varyantları yardım eder ama %100 değil

**Daha sağlam alternatifler:** YOLO / SSD ile plaka tespiti + CRNN tabanlı OCR. Bu klasik pipeline o yöntemler için iyi bir karşılaştırma temelidir.
